In [ ]:
import pandas as pd
import re

df = pd.read_csv("data/updated_data.csv")

# 1. Drop useless columns
df.drop(columns=["Unnamed: 9", "slug"], inplace=True)

# 2. Remove 3 duplicate rows
df.drop_duplicates(inplace=True)

# 3. Fill nulls
df["application"].fillna("Not specified", inplace=True)
df["documents"].fillna("Not specified", inplace=True)
df["tags"].fillna("", inplace=True)

# 4. Clean HTML tags and fix whitespace in all text columns
def clean_text(text):
    if pd.isna(text): return ""
    text = re.sub(r'<[^>]+>', ' ', str(text))     # remove HTML tags
    text = re.sub(r'&[a-z]+;', ' ', text)          # remove &amp; etc
    text = re.sub(r'\s+', ' ', text).strip()       # collapse whitespace
    return text

text_cols = ["scheme_name", "details", "benefits", "eligibility", "application", "documents", "tags"]
for col in text_cols:
    df[col] = df[col].apply(clean_text)

# 5. Normalize category column
df["schemeCategory"] = df["schemeCategory"].str.strip()

# 6. Build full_text column for RAG embedding
def build_full_text(row):
    return (
        f"Scheme: {row['scheme_name']}. "
        f"Level: {row['level']}. "
        f"Category: {row['schemeCategory']}. "
        f"Details: {row['details']}. "
        f"Benefits: {row['benefits']}. "
        f"Eligibility: {row['eligibility']}. "
        f"Application: {row['application']}. "
        f"Documents: {row['documents']}. "
        f"Tags: {row['tags']}."
    )

df["full_text"] = df.apply(build_full_text, axis=1)

# 7. Save
df.to_csv("data/schemes_clean.csv", index=False)
print(f"✅ Clean dataset saved: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.columns.tolist())


ModuleNotFoundError: No module named 'pandas'